<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [يوري كاشنيتسكي](https://yorko.github.io). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center> الموضوع 1. تحليل البيانات الاستكشافية باستخدام الباندا
## <center>الممارسة. تحليل ركاب "تايتانيك". الحل
**املأ الرمز المفقود ("الرمز الخاص بك هنا") واختر الإجابات في [نموذج الويب](https://docs.google.com/forms/d/16EfhpDGPrREry0gfDQdRPjoiQX9IumaL2mPR0rcj19k/edit).**


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 2)
from matplotlib import pyplot as plt

# Graphics in SVG format are more sharp and legible
%config InlineBackend.figure_format = 'svg'


** قراءة البيانات في Pandas DataFrame **


In [ ]:
data = pd.read_csv("../../data/titanic_train.csv", index_col="PassengerId")


** أول 5 صفوف **


In [ ]:
data.head(5)

In [ ]:
data.describe()


** دعنا نختار هؤلاء الركاب الذين صعدوا إلى شيربورج (Embarked=C) ودفعوا > 200 جنيهًا إسترلينيًا مقابل شريطهم (الأجرة > 200).**
تأكد من أنك تفهم كيف يعمل هذا البناء فعليًا.


In [ ]:
data[(data["Embarked"] == "C") & (data.Fare > 200)].head()


**يمكننا تصنيف هؤلاء الأشخاص حسب الأجرة بترتيب تنازلي.**


In [ ]:
data[(data["Embarked"] == "C") & (data["Fare"] > 200)].sort_values(
    by="Fare", ascending=False
).head()


** فلنقم بإنشاء ميزة جديدة.**


In [ ]:
def age_category(age):
    """
    < 30 -> 1
    >= 30, <55 -> 2
    >= 55 -> 3
    """
    if age < 30:
        return 1
    elif age < 55:
        return 2
    elif age >= 55:
        return 3

In [ ]:
age_categories = [age_category(age) for age in data.Age]
data["Age_category"] = age_categories


**هناك طريقة أخرى وهي القيام بذلك باستخدام `apply`.**


In [ ]:
data["Age_category"] = data["Age"].apply(age_category)


**1. كم عدد الرجال/النساء الذين كانوا على متن الطائرة؟**
- 412 رجلاً و479 امرأة
- 314 رجلاً و 577 امرأة
- 479 رجلاً و 412 امرأة
- **<font color='green'>577 رجلاً و314 امرأة [+]</font>**


In [ ]:
(data["Sex"] == "male").sum(), (data["Sex"] == "female").sum()


**أسهل:**


In [ ]:
data["Sex"].value_counts()


**2. اطبع توزيع ميزة `Pclass`. ثم نفس الشيء، ولكن للرجال والنساء بشكل منفصل. كم عدد الرجال من الدرجة الثانية الذين كانوا على متن الطائرة؟**
- 104
- **<font color='green'>108 [+]</font>**
- 112
- 125


In [ ]:
pd.crosstab(data["Pclass"], data["Sex"], margins=True)


يمكننا رسم صورة أيضًا، على الرغم من أنها ليست ضرورية هنا. 


In [ ]:
data["Pclass"].hist(label="all")
data[data["Sex"] == "male"]["Pclass"].hist(color="green", label="male")
data[data["Sex"] == "female"]["Pclass"].hist(color="yellow", label="female")
plt.title("Distribution by class and gender.")
plt.xlabel("Pclass")
plt.ylabel("Frequency")
plt.legend(loc="upper left");

**3. ما هو الانحراف المتوسط ​​والمعياري لـ `Fare`؟. التقريب إلى رقمين عشريين.**
- **<font color='green'>الوسيط هو 14.45، والانحراف المعياري هو 49.69 [+]</font>**
- الوسيط 15.1 والانحراف المعياري 12.15
- الوسيط 13.15 والانحراف المعياري 35.3
- الوسيط 17.43 والانحراف المعياري 39.1


In [ ]:
print("Median fare: ", round(data["Fare"].median(), 2))
print("Fare std: ", round(data["Fare"].std(), 2))


**4. هل هذا صحيح أن متوسط عمر الأشخاص الناجين أعلى من متوسط عمر الركاب الذين ماتوا في النهاية؟**
- نعم
- **<font color='green'>لا [+]</font>**


In [ ]:
data[data["Survived"] == 1]["Age"].hist(
    color="green", label="Survived", alpha=0.5, density=True
)
data[data["Survived"] == 0]["Age"].hist(
    color="red", label="Died", alpha=0.5, density=True
)
plt.title("Age for survived and died")
plt.xlabel("Years")
plt.ylabel("Frequency")
plt.legend();

In [ ]:
#!pip install seaborn
import seaborn as sns

sns.set()

In [ ]:
sns.boxplot(data["Survived"], data["Age"]);


لا يمكن رؤية الفرق من خلال كرة العين فقط. دعونا نحسب.


In [ ]:
data.groupby("Survived")["Age"].mean()


**5. هل هذا صحيح أن الركاب الذين تقل أعمارهم عن 30 عامًا. لقد نجوا بشكل متكرر أكثر من أولئك الذين تزيد أعمارهم عن 60 عامًا؟ ما هي حصص الناجين من الشباب والكبار؟**
- 22.7% بين الشباب و40.6% بين كبار السن
- **<font color='green'>40.6% بين الشباب و22.7% بين كبار السن [+]</font>**
- 35.3% بين الشباب و27.4% بين كبار السن
- 27.4% بين الشباب و35.3% بين كبار السن


In [ ]:
young_survived = data.loc[data["Age"] < 30, "Survived"]
old_survived = data.loc[data["Age"] > 60, "Survived"]

print(
    "Shares of survived people: \n\t  among young {}%, \n\t  among old {}%.".format(
        round(100 * young_survived.mean(), 1), round(100 * old_survived.mean(), 1)
    )
)


**6. هل هذا صحيح أن النساء يبقين على قيد الحياة أكثر من الرجال؟ ما هي حصص الناجين بين الرجال والنساء؟**
- 30.2% بين الرجال و46.2% بين النساء
- 35.7% بين الرجال و 74.2% بين النساء
- 21.1% بين الرجال و46.2% بين النساء
- **<font color='green'>18.9% بين الرجال و 74.2% بين النساء [+]</font>**


In [ ]:
male_survived = data[data["Sex"] == "male"]["Survived"]
female_survived = data[data["Sex"] == "female"]["Survived"]


print(
    "Shares of survived people: \n\t among women {}%, \n\t among men {}%".format(
        round(100 * female_survived.mean(), 1), round(100 * male_survived.mean(), 1)
    )
)


**7. ما هو الاسم الأول الأكثر شيوعًا بين الركاب الذكور؟**
- تشارلز
- توماس
- **<font color='green'>ويليام [+]</font>**
- جون


In [ ]:
data["Name"].head()

In [ ]:
data.loc[1, "Name"].split(",")[1].split()[1]

In [ ]:
first_names = data.loc[data["Sex"] == "male", "Name"].apply(
    lambda full_name: full_name.split(",")[1].split()[1]
)
first_names.value_counts().head()

** 8. كيف يعتمد متوسط ​​عمر الرجال/النساء على `Pclass`؟ اختر جميع العبارات الصحيحة:**
- **<font color='green'>في المتوسط، الرجال من فئة واحدة أكبر من 40 عامًا [+]</font>**
- في المتوسط، النساء من فئة واحدة أكبر من 40 سنة
- **<font color='green'>الرجال من جميع الطبقات هم في المتوسط أكبر سناً من النساء من نفس الفئة [+]</font>**
- **<font color='green'> في المتوسط، ركاب الدرجة الأولى أكبر سناً من ركاب الدرجة الثانية وهم أكبر سناً من ركاب الدرجة الثالثة [+]</font>**


In [ ]:
for cl in data["Pclass"].unique():
    for sex in data["Sex"].unique():
        print(
            "Average age for {0} and class {1}: {2}".format(
                sex,
                cl,
                round(
                    data[(data["Sex"] == sex) & (data["Pclass"] == cl)]["Age"].mean(), 2
                ),
            )
        )


أجمل:


In [ ]:
for (cl, sex), sub_df in data.groupby(["Pclass", "Sex"]):
    print(
        "Average age for {0} and class {1}: {2}".format(
            sex, cl, round(sub_df["Age"].mean(), 2)
        )
    )


وحتى أجمل:


In [ ]:
pd.crosstab(data["Pclass"], data["Sex"], values=data["Age"], aggfunc=np.mean)

In [ ]:
sns.boxplot(data["Pclass"], data["Age"]);


## موارد مفيدة
* نفس دفتر الملاحظات المستخدم على الويب التفاعلي [Kaggle Kernel](https://www.kaggle.com/kashnitsky/topic-1-practice-solution)
* الموضوع الأول "تحليل البيانات الاستكشافية باستخدام الباندا" باعتبارها [Kaggle Kernel](https://www.kaggle.com/kashnitsky/topic-1-exploratory-data-analysis-with-pandas)
* الطبق الرئيسي [الموقع](https://mlcourse.ai)، [مستودع الدورة](https://github.com/Yorko/mlcourse.ai)، ويوتيوب [القناة](https://www.youtube.com/watch?v=QKTuw4PNOsU&list=PLVlY_7IJCMJeRfZ68eVfEcu-UcN9BbwiX)